# E791 \(D^+\to\pi^-\pi^+\pi^+\) — Fit 2 generation example

This notebook generates pseudo-data inspired by **Fit 2** of the Fermilab E791 analysis
(arXiv:hep-ex/0007028v2, 24 Aug 2000).

It uses the Fit 2 central values for the complex coefficients, explicit Bose symmetrization,
`phasespace` weighted MC, and the Laura++ **Covariant** angular formalism requested for
DalitzPlotFitter.

This is therefore a DalitzPlotFitter/Laura++-style realization of the E791 Fit 2
coefficients, not a bit-for-bit recreation of the historical E791 angular convention.

For \(f_0(980)\), the notebook uses the single-channel RBW alternative quoted by E791:
\(m_0=0.975\) GeV, \(\Gamma_0=0.044\) GeV. The companion E791 paper reports that this
alternative gives essentially indistinguishable fractions and phases from the coupled-channel form.


In [ ]:
import numpy as np
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt

from dalitzplotfitter import PhasespaceMC, RealImag, weighted_resample, enable_x64
from dalitzplotfitter.amplitude import (
    AmplitudeComponent,
    BoseSymmetrizedAmplitude,
    CoherentAmplitudeModel,
    ConstantAmplitude,
)
from dalitzplotfitter.dynamics import LauraCovariantRBW

enable_x64()


## 1. Masses and Fit 2 coefficients

Ordering:

\[
p_1=\pi^-,\qquad p_2=\pi^+_1,\qquad p_3=\pi^+_2.
\]

The \(\rho^0(770)\pi^+\) amplitude is the fixed reference, \(c_\rho=1+0i\).


In [ ]:
M_D = 1.86966
M_PI = 0.13957039

resonances = {
    "sigma":   dict(mass=0.478,  width=0.324, spin=0),
    "rho770":  dict(mass=0.7693, width=0.1502, spin=1),
    "f0_980":  dict(mass=0.975,  width=0.044, spin=0),
    "f2_1270": dict(mass=1.275,  width=0.185, spin=2),
    "f0_1370": dict(mass=1.434,  width=0.173, spin=0),
    "rho1450": dict(mass=1.465,  width=0.310, spin=1),
}

fit2_polar = {
    "sigma":   (1.17, 205.7),
    "rho770":  (1.00,   0.0),
    "NR":      (0.48,  57.3),
    "f0_980":  (0.43, 165.0),
    "f2_1270": (0.76,  57.3),
    "f0_1370": (0.26, 105.4),
    "rho1450": (0.14, 319.1),
}

def polar_to_xy(magnitude, phase_deg):
    phase = np.deg2rad(phase_deg)
    return magnitude * np.cos(phase), magnitude * np.sin(phase)

fit2_xy = {name: polar_to_xy(*value) for name, value in fit2_polar.items()}

for name, (x, y) in fit2_xy.items():
    mag, phase = fit2_polar[name]
    print(f"{name:8s}: r={mag:5.2f}, phi={phase:6.1f} deg -> x={x:+.5f}, y={y:+.5f}")


## 2. Bose-symmetrized Covariant resonance amplitudes

For each resonance:

\[
F_i = F_i[(12)3] + F_i[(13)2].
\]

The unlike-charge pion is kept as the selected resonance daughter in both pairings.


In [ ]:
R_PARENT = 3.0
R_RESONANCE = 3.0

def symmetrized_rbw(name):
    pars = resonances[name]
    common = dict(
        mass0=pars["mass"],
        width0=pars["width"],
        parent_mass=M_D,
        daughter_masses=(M_PI, M_PI),
        bachelor_mass=M_PI,
        angular_momentum=pars["spin"],
        resonance_radius=R_RESONANCE,
        parent_radius=R_PARENT,
    )
    pair12 = LauraCovariantRBW(
        **common,
        daughter_key="p1",
        partner_key="p2",
        bachelor_key="p3",
    )
    pair13 = LauraCovariantRBW(
        **common,
        daughter_key="p1",
        partner_key="p3",
        bachelor_key="p2",
    )
    return BoseSymmetrizedAmplitude(pair12, pair13)

components = []
for name in ("sigma", "rho770", "f0_980", "f2_1270", "f0_1370", "rho1450"):
    x, y = fit2_xy[name]
    components.append(AmplitudeComponent(name, symmetrized_rbw(name), RealImag(x, y)))

x_nr, y_nr = fit2_xy["NR"]
components.append(AmplitudeComponent("NR", ConstantAmplitude(), RealImag(x_nr, y_nr)))

model = CoherentAmplitudeModel(tuple(components))


## 3. Generate weighted phase-space candidates

The target importance weight is

\[
w_k^{\rm target}=w_k^{\rm PS}|A(x_k)|^2.
\]


In [ ]:
N_POOL = 1_000_000
N_TOY = 100_000

generator = PhasespaceMC(M_D, (M_PI, M_PI, M_PI))
pool = generator.generate(N_POOL, seed=2000)
data_pool = pool.as_dict()

amplitude_pool = model.amplitude(data_pool)
intensity_pool = jnp.abs(amplitude_pool) ** 2
target_weights = pool.weights * intensity_pool

print("candidate pool:", pool.size)
print("finite target weights:", bool(jnp.all(jnp.isfinite(target_weights))))
print("sum target weights:", float(jnp.sum(target_weights)))


## 4. Raw phase space


In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
ax.hist2d(np.asarray(pool.s12), np.asarray(pool.s13), bins=120,
          weights=np.asarray(pool.weights))
ax.set_xlabel(r"$s_{12}$ [GeV$^2$]")
ax.set_ylabel(r"$s_{13}$ [GeV$^2$]")
ax.set_title("Weighted three-body phase space")
plt.show()


## 5. Fit 2 weighted Dalitz density


In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
ax.hist2d(np.asarray(pool.s12), np.asarray(pool.s13), bins=120,
          weights=np.asarray(target_weights))
ax.set_xlabel(r"$s_{12}$ [GeV$^2$]")
ax.set_ylabel(r"$s_{13}$ [GeV$^2$]")
ax.set_title("E791 Fit 2-inspired weighted Dalitz density")
plt.show()


## 6. Resample 100k unweighted pseudo-data events


In [ ]:
toy = weighted_resample(
    jax.random.key(791),
    pool,
    target_weights,
    N_TOY,
    replace=True,
)

print("toy events:", toy.size)
print("unique toy weights:", np.unique(np.asarray(toy.weights)))


In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
ax.hist2d(np.asarray(toy.s12), np.asarray(toy.s13), bins=100)
ax.set_xlabel(r"$s_{12}$ [GeV$^2$]")
ax.set_ylabel(r"$s_{13}$ [GeV$^2$]")
ax.set_title("100k unweighted Fit 2-inspired pseudo-data")
plt.show()


## 7. Symmetrized \(\pi^+\pi^-\) projection

As in E791, fill both \(\pi^+\pi^-\) combinations.


In [ ]:
s_pm = np.concatenate([np.asarray(toy.s12), np.asarray(toy.s13)])
fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(s_pm, bins=100, histtype="step", linewidth=1.6)
ax.set_xlabel(r"$m^2(\pi^+\pi^-)$ [GeV$^2$]")
ax.set_ylabel("Entries / bin")
ax.set_title(r"Fit 2-inspired $s_{12}+s_{13}$ projection")
plt.show()


## 8. Individual component intensities

These curves do **not** sum to the coherent total because interference is omitted in this diagnostic.


In [ ]:
bins = np.linspace(
    float(min(jnp.min(pool.s12), jnp.min(pool.s13))),
    float(max(jnp.max(pool.s12), jnp.max(pool.s13))),
    100,
)
s_both = np.concatenate([np.asarray(pool.s12), np.asarray(pool.s13)])

fig, ax = plt.subplots(figsize=(9, 6))
for component in components:
    amp_i = component.value(data_pool)
    w_i = np.asarray(pool.weights * jnp.abs(amp_i) ** 2)
    w_both = np.concatenate([w_i, w_i])
    hist, edges = np.histogram(s_both, bins=bins, weights=w_both)
    if hist.sum() > 0:
        hist = hist / hist.sum()
    centers = 0.5 * (edges[:-1] + edges[1:])
    ax.plot(centers, hist, label=component.name)

ax.set_xlabel(r"$m^2(\pi^+\pi^-)$ [GeV$^2$]")
ax.set_ylabel("Normalized component intensity")
ax.set_title("Fit 2 component shapes")
ax.legend(ncol=2)
plt.show()


## 9. Net interference


In [ ]:
coherent = np.asarray(intensity_pool)
incoherent = np.zeros(pool.size)
for component in components:
    amp_i = np.asarray(component.value(data_pool))
    incoherent += np.abs(amp_i) ** 2

interference = coherent - incoherent

fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(np.asarray(pool.s12), bins=100,
        weights=np.asarray(pool.weights) * interference,
        histtype="step")
ax.set_xlabel(r"$s_{12}$ [GeV$^2$]")
ax.set_ylabel("Weighted net interference")
ax.set_title("Net interference projection")
plt.show()


## Fit 2 central values used

| Mode | Magnitude | Phase (deg) | Fit fraction (%) |
|---|---:|---:|---:|
| \(\sigma\pi^+\) | 1.17 | 205.7 | 46.3 |
| \(\rho^0(770)\pi^+\) | 1.00 | 0.0 | 33.6 |
| NR | 0.48 | 57.3 | 7.8 |
| \(f_0(980)\pi^+\) | 0.43 | 165.0 | 6.2 |
| \(f_2(1270)\pi^+\) | 0.76 | 57.3 | 19.4 |
| \(f_0(1370)\pi^+\) | 0.26 | 105.4 | 2.3 |
| \(\rho^0(1450)\pi^+\) | 0.14 | 319.1 | 0.7 |

The fit fractions do not need to sum to 100% because the amplitudes interfere.

Reference: E. M. Aitala *et al.* (E791), arXiv:hep-ex/0007028v2.
